In [27]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from pathlib import Path
import textwrap
import cga_utils
import math
from typing import Iterable, List, Optional, Tuple
import hdbscan
from sklearn.feature_extraction import DictVectorizer

In [52]:
def err_bucket(rel_err):
    if rel_err < 1e-6: 
        return 0
    elif rel_err < 0.01:
        return 1
    elif rel_err < 0.1:
        return 2
    elif rel_err < 1:
        return 3
    else:
        return 4


def prepare_matrix(errors):
    EPS = 1e-9
    
    errors["rel_err"] = errors.apply(lambda row: (row["pred"] - row["answer"]) / max(row["answer"], EPS), axis=1  )
    errors["abs_err"] = errors.apply(lambda row: row["pred"] - row["answer"], axis=1  )
    errors["magnitude_bucket"] = errors.apply(lambda row: int(math.floor(math.log10(abs(row["answer"])+EPS))) if row["answer"] != 0 else -1, axis=1  )
    errors["rel_error_bucket"] = errors["rel_err"].apply(err_bucket)
    errors["ratio"] = errors.apply(lambda row: (row["pred"] / (row["answer"]+EPS)) if row["answer"] != 0 else np.inf,  axis=1 )
    errors["x100_flag"] = errors["ratio"].apply(lambda ratio: int(0.95 < ratio/100 < 1.05 or 0.95 < ratio*100 < 1.05))
    errors["has_error_text"] = errors["error_text"].apply(lambda txt: txt != None and txt != '')
    errors["ratio_is_inf"] = np.isinf(errors["ratio"]).astype(int)
    errors["ratio"] = np.where(np.isinf(errors["ratio"]), np.nan, errors["ratio"])

    # 2) Erősen ferde oszlopok log1p-vel és/vagy winsorize/clip
    for col in ["abs_err", "rel_err", "ratio"]:
        if col in errors:
            # negatív is lehet -> signed log1p
            errors[col] = np.sign(errors[col]) * np.log1p(np.abs(errors[col]))

    # 3) NaN-ek kezelése (pl. median impute)
    num_cols = ["rel_err","abs_err","ratio"]
    for col in num_cols:
        if col in errors:
            med = errors[col].median()
            errors[col] = errors[col].fillna(med)

    # 4) Skálázás a folytonosokra (RobustScaler a kilógók ellen)
    from sklearn.preprocessing import RobustScaler
    cont = errors[num_cols].values
    scaler = RobustScaler().fit(cont)
    errors[num_cols] = scaler.transform(cont)

    errors["scale"] = errors["scale"].fillna("")
    errors["pred_scale"] = errors["pred_scale"].fillna("")
    errors["code_calc_pattern"]= errors["code_calc_pattern"].fillna("")

    #vects = errors.drop(["ts", "qid", "pred_scale", "question", "derivation", "calc_pattern", "pred", "answer", "scale", "value_match", "error_text",	"value_list", "code",	"selected_values","needed_values","exact_match",	"error_code"], axis=1)
    #vects = errors[["calc_pattern", "error_code",  "pred_ast", "selection_success", "sign_error", "is_parenth_in_table", "has_code_abs",
    # "scale", "pred_scale",  "rel_error_bucket", "magnitude_bucket", "x100_flag",  "has_error_text"]]
    vects = errors[["calc_pattern", "code_calc_pattern", "scale", "pred_scale"]]
    
    X_dict = vects.to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    X_flags = dv.fit_transform(X_dict)   
    return X_flags

In [58]:

def cluster_hdbscan(X: np.ndarray, min_cluster_size: int = 8, min_samples: int = 2) -> Tuple[np.ndarray, Optional[float]]:
    if hdbscan is None:
        print("[warn] hdbscan not installed; falling back to agglomerative.")
        return cluster_agglomerative(X, min_cluster_size)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric="manhattan")
    labels = clusterer.fit_predict(X)
    return labels, None
    from typing import Iterable, List, Optional, Tuple

def try_hdbscan(X, mcs_list=(2,3,4, 5,6, 10,15), ms_list=(1,2,3, 4, 5)):
    from collections import defaultdict
    out = []
    for mcs in mcs_list:
        for ms in ms_list:
            cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms, metric="manhattan")
            labels = cl.fit_predict(X)
            n_clusters = len(set(labels) - {-1})
            noise_ratio = (labels == -1).mean()
            out.append((mcs, ms, n_clusters, noise_ratio))        
    return pd.DataFrame(out).sort_values(by=3)

# Usage

In [59]:
errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')
X_features = prepare_matrix(errors)
try_hdbscan(X_features)

,0,1,2,3
0,2,1,46,0.117647
15,5,1,11,0.203209
5,3,1,27,0.219251
30,15,1,3,0.224599
26,10,2,4,0.224599
20,6,1,10,0.229947
10,4,1,18,0.235294
28,10,4,3,0.251337
33,15,4,3,0.251337
1,2,2,31,0.267380


In [ ]:
labels, thr = cluster_hdbscan(X_features, min_cluster_size=4, min_samples=1)

In [60]:
(labels == -1).mean()

0.23529411764705882

In [61]:
labels

array([ 7,  7, -1,  7, -1, -1,  1, -1,  1,  1,  9, 17, 12, 10, -1,  5,  5,
       -1,  2, -1, -1, -1, 16, -1,  3,  3, 16, 16, 11, 16, -1, -1, 15, 15,
       13,  4, 16,  4, 13, 11,  6, 11, 17,  0,  0,  9, 16, -1,  4,  9,  9,
        8,  3,  3,  4, -1, -1, -1, 10, 12, 16,  8, -1, -1,  9,  9, -1,  0,
       11, -1, -1, -1,  8,  0,  0,  0, -1, 14,  2, 13,  1,  1,  3, 17,  5,
       11, -1, -1, 10, 10, -1, -1, 17,  8, 11, 11, 16, -1,  9, 17, 16, 16,
        2,  2,  1,  1,  1, -1, 16, 16, -1, 16,  1, -1,  1, 16, 12, 12, 12,
       15, -1,  9, 16, 12, -1, 16, -1,  9,  7,  7,  9, 10, 10, -1, 11, -1,
        0, -1, -1,  6, 16, 17, 14, 10,  7,  4,  0,  0, 11, -1, -1, 16, 16,
       15, -1,  3,  4, 13, 12,  4, 11,  1, 14,  0,  6,  6, -1,  4,  3, 10,
       10, 13,  5, 11,  7, 11, 14, 17,  8, 16,  4,  3, -1, -1, 13, 13, 13])

In [63]:
errors["labels"] = labels

In [64]:
errors["labels"].value_counts()

labels
-1     44
 16    20
 11    12
 1     11
 9     10
 0     10
 4      9
 10     9
 3      8
 13     8
 7      7
 12     7
 17     7
 8      5
 2      4
 15     4
 5      4
 6      4
 14     4
Name: count, dtype: int64

In [71]:
errors = pd.read_csv('res/e38_23.csv').query('exact_match == False')
X_features = prepare_matrix(errors)
try_hdbscan(X_features)

,0,1,2,3
34,15,5,2,0.000000
29,10,5,2,0.000000
28,10,4,2,0.113636
27,10,3,2,0.113636
25,10,1,2,0.142045
31,15,2,2,0.198864
26,10,2,2,0.198864
0,2,1,39,0.204545
33,15,4,2,0.210227
5,3,1,28,0.250000


In [74]:
labels, thr = cluster_hdbscan(X_features, min_cluster_size=4, min_samples=1)
errors["labels"] = labels
errors["labels"].value_counts()

labels
-1     51
 12    10
 15     9
 14     8
 0      8
 5      7
 19     7
 13     7
 10     7
 8      6
 3      6
 7      6
 4      5
 2      5
 6      5
 1      5
 17     4
 16     4
 9      4
 11     4
 20     4
 18     4
Name: count, dtype: int64

In [80]:
[(i["question"], i["calc_pattern"], i["code_calc_pattern"]) for idx, i in errors.iterrows() if i["labels"] == 14]

[('What is the difference in amount between Deferred Revenue and Other non-current liabilities as reported?',
  '#-#',
  '#-#'),
 ('What was the increase / (decrease) in the Statutory federal income tax (benefit) from 2018 to 2019?',
  '#-#',
  '#-#'),
 ('What was the change in Inventory between 2018 and 2019?', '#-#', '#-#'),
 ('What is the increase / (decrease) in the Gross Profit from 2018 to 2019?',
  '#-#',
  '#-#'),
 ('What is the COGS for 2019?', '#-#', '#-#'),
 ('What was the change in Impairment losses between 2017 and 2018?',
  '#-#',
  '#-#'),
 ('What was the change in Net operating loss carryforwards from 2018 to 2019?',
  '#-#',
  '#-#'),
 ('What was the change in the number of shares granted in 2019 from 2018?',
  '#-#',
  '#-#')]

In [81]:
errors = pd.read_csv('res/e38_34b.csv').query('exact_match == False')
X_features = prepare_matrix(errors)
try_hdbscan(X_features)

,0,1,2,3
31,15,2,2,0.082840
26,10,2,2,0.082840
30,15,1,2,0.088757
25,10,1,2,0.088757
28,10,4,2,0.177515
33,15,4,2,0.177515
16,5,2,11,0.177515
21,6,2,8,0.195266
0,2,1,39,0.201183
32,15,3,2,0.218935


In [82]:
labels, thr = cluster_hdbscan(X_features, min_cluster_size=4, min_samples=1)
errors["labels"] = labels
errors["labels"].value_counts()

labels
-1     48
 1     12
 4     10
 13    10
 5     10
 16     9
 10     8
 7      7
 11     6
 2      6
 15     6
 14     5
 3      5
 12     5
 8      5
 0      5
 6      4
 9      4
 17     4
Name: count, dtype: int64

In [88]:
[(i["question"], i["calc_pattern"], i["code_calc_pattern"]) for idx, i in errors.iterrows() if i["labels"] == 16]

[('What is the 2019 average defined contribution schemes?', '(#+#)/#', '#/#'),
 ('What is the 2019 average defined benefit schemes?', '(#+#)/#', '#/#'),
 ('What is the 2018 average free cash flow?', '(#+#)/#', '#'),
 ('What is the average total current tax expense for 2017 and 2018? ',
  '(#+#)/#',
  '(#+#)/#'),
 ('What is the average total current tax expense for 2018 and 2019?',
  '(#+#)/#',
  '(#+#)/#'),
 ('What is the average total asset value for 2018 and 2019?',
  '(#+#)/#',
  '(#+#)/#'),
 ('What is the 2019 average total amount falling due within one year?',
  '(#+#)/#',
  '#'),
 ('What is the 2019 average total amount falling due after more than one year?',
  '(#+#)/#',
  '(#+#)/#'),
 ('What is the average unvested restricted stock?', '(#+#)/#', '#/#')]

In [93]:
errors = pd.read_csv('res/e38_35.csv').query('exact_match == False')
X_features = prepare_matrix(errors)
try_hdbscan(X_features)

,0,1,2,3
26,10,2,2,0.000000
27,10,3,2,0.000000
28,10,4,2,0.000000
25,10,1,2,0.019231
29,10,5,2,0.102564
0,2,1,41,0.166667
30,15,1,2,0.173077
1,2,2,31,0.211538
32,15,3,2,0.211538
10,4,1,14,0.250000


In [95]:
labels, thr = cluster_hdbscan(X_features, min_cluster_size=4, min_samples=1)
errors["labels"] = labels
errors["labels"].value_counts()

labels
-1     39
 7     16
 13    14
 11    12
 8     11
 0     10
 2     10
 12     8
 3      8
 6      7
 5      5
 10     4
 1      4
 4      4
 9      4
Name: count, dtype: int64

In [96]:
[(i["question"], i["calc_pattern"], i["code_calc_pattern"]) for idx, i in errors.iterrows() if i["labels"] == 7]

[('What is the 2019 average defined contribution schemes?', '(#+#)/#', '#/#'),
 ('What is the 2019 average defined benefit schemes?', '(#+#)/#', '#/#'),
 ('What is the 2019 average free cash flow?', '(#+#)/#', '#'),
 ('What is the 2018 average free cash flow?', '(#+#)/#', ''),
 ('What was the change in closing cash?', '#-#', ''),
 ('What is the average total current tax expense for 2017 and 2018? ',
  '(#+#)/#',
  '(#+#)/#'),
 ('What is the average total asset value for 2018 and 2019?',
  '(#+#)/#',
  '(#+#)/#'),
 ('What is the Total net revenue for fiscal 2018 and 2017?', '#+#', '#'),
 ('What is the total carrying amount of Senior Notes due by December 2024 as of December 31, 2019?',
  '#+#+#+#',
  '#'),
 ('What is the 2019 average total amount falling due within one year?',
  '(#+#)/#',
  '#'),
 ('What is the 2019 average total amount falling due after more than one year?',
  '(#+#)/#',
  '(#+#)/#'),
 ('What is the average unvested restricted stock?', '(#+#)/#', '#/#'),
 ('What is th

In [98]:
errors = pd.read_csv('res/e38_36.csv').query('exact_match == False')
X_features = prepare_matrix(errors)
try_hdbscan(X_features)

FileNotFoundError: [Errno 2] No such file or directory: 'res/e38_36.csv'

In [99]:
labels, thr = cluster_hdbscan(X_features, min_cluster_size=4, min_samples=1)
errors["labels"] = labels
errors["labels"].value_counts()

labels
-1     39
 7     16
 13    14
 11    12
 8     11
 0     10
 2     10
 12     8
 3      8
 6      7
 5      5
 10     4
 1      4
 4      4
 9      4
Name: count, dtype: int64

In [101]:
pd.DataFrame([(i["question"], i["calc_pattern"], i["code_calc_pattern"]) for idx, i in errors.iterrows() if i["labels"] == 7])

,0,1,2
0,What is the 2019 average defined contribution ...,(#+#)/#,#/#
1,What is the 2019 average defined benefit schemes?,(#+#)/#,#/#
2,What is the 2019 average free cash flow?,(#+#)/#,#
3,What is the 2018 average free cash flow?,(#+#)/#,
4,What was the change in closing cash?,#-#,
5,What is the average total current tax expense ...,(#+#)/#,(#+#)/#
6,What is the average total asset value for 2018...,(#+#)/#,(#+#)/#
7,What is the Total net revenue for fiscal 2018 ...,#+#,#
8,What is the total carrying amount of Senior No...,#+#+#+#,#
9,What is the 2019 average total amount falling ...,(#+#)/#,#


In [104]:
errors[["question","calc_pattern","code_calc_pattern", "labels"]].query('labels == 7')

,question,calc_pattern,code_calc_pattern,labels
5,What is the 2019 average defined contribution ...,(#+#)/#,#/#,7
6,What is the 2019 average defined benefit schemes?,(#+#)/#,#/#,7
14,What is the 2019 average free cash flow?,(#+#)/#,#,7
15,What is the 2018 average free cash flow?,(#+#)/#,,7
38,What was the change in closing cash?,#-#,,7
39,What is the average total current tax expense ...,(#+#)/#,(#+#)/#,7
232,What is the average total asset value for 2018...,(#+#)/#,(#+#)/#,7
248,What is the Total net revenue for fiscal 2018 ...,#+#,#,7
334,What is the total carrying amount of Senior No...,#+#+#+#,#,7
340,What is the 2019 average total amount falling ...,(#+#)/#,#,7
